In [7]:
import kaggle_benchmarks as kbench
import pandas as pd
import io
import requests
import string

TEST_URL = "https://raw.githubusercontent.com/ucl-dark/ludwig/main/test_conversational_implicatures.csv"

@kbench.task(
    name="Pragmatic Intent Detection (LUDWIG)",
    description="Evaluates social cognition by detecting implied 'yes' or 'no' responses over 250 samples."
)
def ludwig_benchmark_task(llm) -> float: # Note: We declare this returns a float
    # 1. Fetch Data
    raw_csv = requests.get(TEST_URL).content
    df = pd.read_csv(io.StringIO(raw_csv.decode('utf-8')))
    
    # 2. Filter for strictly binary implicatures
    def get_binary_label(text):
        text = str(text).lower().strip()
        if text.startswith("yes"): return "yes"
        if text.startswith("no"): return "no"
        return None

    df['target'] = df['Implicature'].apply(get_binary_label)
    df = df.dropna(subset=['target'])
    
    # 3. Fixed Seed
    samples = df.sample(n=250, random_state=42)
    correct_count = 0

    for i, row in samples.iterrows():
        # 4. Construct the Prompt
        prompt = f"""
        Speaker A: "{row['Context utterance']}"
        Speaker B: "{row['Response utterance']}"
        Question: Based on Speaker B's response, is the answer 'yes' or 'no'? 
        Respond with ONLY the word 'yes' or the word 'no'.
        """

        # 5. Model Inference & Cleaning
        response = llm.prompt(prompt).strip().lower()
        response = response.translate(str.maketrans('', '', string.punctuation))
        expected = row['target']

        # 6. Safe Scoring (No Assertions)
        if response == expected:
            correct_count += 1
        else:
            # Logs the failure to your console so you can still review mistakes
            print(f"Sample {i} Failed | Expected: '{expected}', Got: '{response}'")

    # 7. Calculate and return the final metric
    final_score = correct_count / len(samples)
    return final_score

# Execute the task
ludwig_benchmark_task.run(kbench.llm)

BokehModel(combine_events=True, render_bundle={'docs_json': {'dff4a833-5740-4227-9343-eff88e11ec02': {'version…